> 在线讲义：[第 44 章 · 训练稳定性：logit 漂移与 z-loss、QK-norm、梯度裁剪、低精度格式与缩放因子](https://wikieden.github.io/math-and-eval-stats/lessons/44-training-stability.html) · 本书《数学与实验统计》

# 第 44 章 · 训练稳定性：logit 漂移与 z-loss、QK-norm、梯度裁剪、低精度格式与缩放因子

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/44-training-stability.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 z-loss | c(log Z)² | `ce_and_zloss(z + 50, 0)` | CE 0.4402、z-loss 0.275 |
| 第二步 QK-norm | g²√d | `qk_logit_bound(128, 1.0)` | 11.31 |
| 第三步 裁剪 | 全局范数 | `clip_global(grads, 1.0)` | 13 → 1 |
| 第四步 格式 | 表 | `FORMATS` | 448、57344 |
| 第五步 fp8 缩放 | amax/448 | `fp8_scale(x)`、`round_e4m3` | 误差 0.60 → 0.023 |

例子：一组 logits 整体漂高时 softmax 不变、z-loss 会把它拉回来；QK-norm 给注意力 logit 一个上界；按全局范数裁剪梯度；
bf16 / fp16 / fp8 的数值范围；fp8 训练的 per-tensor 缩放因子。
里程碑 ME-M27 的“画 bf16 与 fp8 的可表示范围”留给自己做；这里只列数。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

**函数 `logsumexp`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def logsumexp(z):
    m = np.max(z)
    return float(m + np.log(np.exp(z - m).sum()))

**函数 `ce_and_zloss`**：交叉熵 = logsumexp(z) − z_target；z-loss = coef·(log Z)²（ST-MoE 式 (5)），把 log Z 拉向 0。

In [ ]:
def ce_and_zloss(z: np.ndarray, target: int, coef: float = 1e-4) -> tuple[float, float]:
    """交叉熵 = logsumexp(z) − z_target；z-loss = coef·(log Z)²（ST-MoE 式 (5)），把 log Z 拉向 0。"""
    lz = logsumexp(z)
    return lz - float(z[target]), coef * lz * lz

**函数 `qk_logit_bound`**：QK-norm：q、k 先各自 RMSNorm（长度 √d）再乘可学的 g，点积 / √d 的绝对值 ≤ g²·d·… 这里按长度 √d 的两个向量：

In [ ]:
def qk_logit_bound(d: int, g: float) -> float:
    """QK-norm：q、k 先各自 RMSNorm（长度 √d）再乘可学的 g，点积 / √d 的绝对值 ≤ g²·d·… 这里按长度 √d 的两个向量：
    |q·k|/√d ≤ √d·√d/√d·g² = g²√d。"""
    return g * g * np.sqrt(d)

**函数 `clip_global`**：全局范数裁剪：所有参数的梯度拼成一个向量求范数，超过 max_norm 就整体乘 max_norm/范数（方向不变）。

In [ ]:
def clip_global(grads: list[np.ndarray], max_norm: float) -> tuple[float, list[np.ndarray]]:
    """全局范数裁剪：所有参数的梯度拼成一个向量求范数，超过 max_norm 就整体乘 max_norm/范数（方向不变）。"""
    total = float(np.sqrt(sum((g ** 2).sum() for g in grads)))
    s = min(1.0, max_norm / (total + 1e-12))
    return total, [g * s for g in grads]

In [ ]:
FORMATS = {   # 名字：(指数位, 尾数位, 最大值, 最小正规数)
    "fp32": (8, 23, 3.4028e38, 1.1755e-38),
    "bf16": (8, 7, 3.3895e38, 1.1755e-38),
    "fp16": (5, 10, 65504.0, 6.1035e-05),
    "fp8 E4M3": (4, 3, 448.0, 2 ** -6),
    "fp8 E5M2": (5, 2, 57344.0, 2 ** -14),
}

**函数 `fp8_scale`**：per-tensor 缩放：scale = amax / fmt_max，存 x/scale（落进 E4M3 的范围），用时再乘回去。

In [ ]:
def fp8_scale(x: np.ndarray, fmt_max: float = 448.0) -> float:
    """per-tensor 缩放：scale = amax / fmt_max，存 x/scale（落进 E4M3 的范围），用时再乘回去。"""
    return float(np.abs(x).max() / fmt_max)

**函数 `round_e4m3`**：模拟 E4M3：正规数保留 3 位尾数（就近），截到 ±448；|x| < 2^−6 时进入次正规数，间距固定为 2^−9（更小的直接舍成 0）。

In [ ]:
def round_e4m3(x: np.ndarray) -> np.ndarray:
    """模拟 E4M3：正规数保留 3 位尾数（就近），截到 ±448；|x| < 2^−6 时进入次正规数，间距固定为 2^−9（更小的直接舍成 0）。"""
    x = np.clip(x, -448, 448)
    e = np.floor(np.log2(np.maximum(np.abs(x), 1e-30)))
    q = 2.0 ** (np.maximum(e, -6) - 3)
    return np.round(x / q) * q

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 logit 漂移：softmax 看不见，z-loss 看得见")
z = np.array([2.0, 1.0, 0.0, -1.0])
for shift in (0, 20, 50):
    ce, zl = ce_and_zloss(z + shift, 0)
    print(f"  logits + {shift}：交叉熵 {ce:.4f}（不变），log Z = {logsumexp(z + shift):.2f}，z-loss = {zl:.4f}")

In [ ]:
print("\n第二步 QK-norm：注意力 logit 有了上界")
for g in (1.0, 2.0):
    print(f"  d = 128、增益 g = {g}：|q·k|/√d ≤ {qk_logit_bound(128, g):.2f}")

In [ ]:
print("\n第三步 梯度裁剪：按全局范数整体缩放")
grads = [np.array([3.0, 4.0]), np.array([12.0])]
tot, cl = clip_global(grads, 1.0)
print(f"  全局范数 {tot:.1f} → 裁到 1：{[np.round(c, 4).tolist() for c in cl]}（方向不变）")

In [ ]:
print("\n第四步 低精度格式的范围")
for k, (e, m, mx, mn) in FORMATS.items():
    print(f"  {k}：指数 {e} 位、尾数 {m} 位，最大 {mx:.4g}，最小正规数 {mn:.3g}，相对精度 2^-{m} = {2 ** -m:.3g}")

In [ ]:
print("\n第五步 fp8 训练：先按 amax 缩放再量化")
rng = np.random.default_rng(0)
x = rng.normal(0, 0.001, 4096)
x[0] = 0.05
s = fp8_scale(x)
direct = round_e4m3(x)
scaled = round_e4m3(x / s) * s
print(f"  梯度标准差 0.001、最大 0.05：scale = {s:.3e}")
print(f"  不缩放直接转 E4M3：{(direct == 0).mean():.0%} 的值变成 0，相对误差 {np.abs(direct - x).mean() / np.abs(x).mean():.4f}，缩放后 {np.abs(scaled - x).mean() / np.abs(x).mean():.4f}")

## 3. 代码里才看得到的

1. **E4M3 的模拟是近似的**：只处理了尾数舍入、次正规区和 ±448 截断，没有处理 NaN 编码和就近偶数舍入的细节。
2. **z-loss 系数很小**：c_z = 10⁻⁴，正常 logits 下几乎不影响主损失，只在 log Z 变大时起作用。
3. **没有画格式的数轴**：E4M3 / E5M2 / bf16 的可表示范围图留给自己画（里程碑 ME-M27）。

## 4. 自己改一改

1. 用 E5M2（尾数 2 位、最小正规数 2⁻¹⁴）重写 `round_e4m3`，不缩放时还有多少值变成 0？
2. 把 `clip_global` 改成逐元素裁剪到 [−1, 1]，比较裁剪后梯度和原梯度的夹角。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> softmax 看不见整体漂移，z-loss 罚 log Z 的平方；
> QK-norm 先归一，logit 上界 g 方乘根号 d；
> 梯度按全局范数裁，只缩长度不改方向；
> 指数管范围、尾数管精度，fp8 一定要缩放。